# Anomaly detection for VLST (stent thrombosis)

One-class anomaly detection baselines trained **only on patients without stent thrombosis** (class 0), compared against supervised TabPFN from [tabpfn.ipynb](tabpfn.ipynb) on the **same 70/30 stratified split**.

Methods:
- **Isolation Forest** — isolates rare points in feature space
- **One-Class SVM** — boundary around the normal population
- **Autoencoder** — high reconstruction error flags anomalies

Preprocessing: drop ID/leakage columns (`NO.`, `Name`, `Time since stent implantation`), drop rows with any missing value (no imputation).

**Paths:** auto-detects local repo vs Kaggle. On Kaggle attach `vlst-data` and `preprocessed-data` datasets (same paths as `tabpfn.ipynb`).


In [ ]:
import os
import time
import warnings
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
np.random.seed(42)

is_kaggle_env = os.path.isdir("/kaggle/working")
# is_kaggle_env = True  # uncomment if auto-detect fails
print(f"Runtime: {'Kaggle' if is_kaggle_env else 'local'}")


In [ ]:
TARGET_COL = "Stent thrombosis"
DROP_FEATURES = ["Time since stent implantation"]
ID_COLS = ["NO.", "Name"]
RANDOM_STATE = 42
TEST_SIZE = 0.3
VAL_SIZE = 0.15  # fraction of train for hyperparameter tuning

KAGGLE_RAW_CSV = "/kaggle/input/datasets/amirmahdidaraei/vlst-data/VLST.csv"
KAGGLE_PROCESSED_DIR = "/kaggle/input/datasets/amirmahdidaraei/preprocessed-data"
KAGGLE_RESULT_SUBDIR = "anomaly_detection"


def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "data" / "raw" / "VLST.csv").is_file():
            return d
    raise FileNotFoundError(
        "Could not locate data/raw/VLST.csv above the current working directory."
    )


def _discover_vlst_csv() -> Path:
    """Resolve VLST.csv on Kaggle (env override, then known dataset paths)."""
    env = os.environ.get("VLST_RAW_CSV")
    if env and Path(env).is_file():
        return Path(env)

    candidates = [
        Path(KAGGLE_RAW_CSV),
        Path("/kaggle/input/datasets/amirmahdidaraei/vlst-data/VLST.csv"),
        Path("/kaggle/input/vlst-data/VLST.csv"),
        Path("/kaggle/input/VLST_data/VLST.csv"),
    ]
    for p in candidates:
        if p.is_file():
            return p

    base = Path("/kaggle/input")
    if base.is_dir():
        for p in base.rglob("VLST.csv"):
            if p.is_file():
                return p

    raise FileNotFoundError(
        "VLST.csv not found on Kaggle. Set VLST_RAW_CSV or attach the vlst-data dataset."
    )


def _resolve_paths():
    """Return (raw_path, processed_dir, result_dir, label) for local or Kaggle."""
    if is_kaggle_env:
        raw = _discover_vlst_csv()
        processed = Path(os.environ.get("VLST_PROCESSED_DIR", KAGGLE_PROCESSED_DIR))
        result = Path(
            os.environ.get(
                "VLST_ANOMALY_RESULT_DIR",
                str(Path("/kaggle/working") / KAGGLE_RESULT_SUBDIR),
            )
        )
        return raw, processed, result, f"Kaggle | raw={raw}"

    repo = _find_repo_root()
    return (
        repo / "data" / "raw" / "VLST.csv",
        repo / "data" / "processed",
        repo / "data" / "result" / "anomaly_detection",
        f"local | repo={repo}",
    )


RAW_PATH, PROCESSED_DIR, RESULT_DIR, _path_label = _resolve_paths()
RAW_PATH = Path(RAW_PATH)
PROCESSED_DIR = Path(PROCESSED_DIR)
RESULT_DIR = Path(RESULT_DIR)
COL_CONFIG = PROCESSED_DIR / "column_config.csv"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(_path_label)
print("RAW_PATH:", RAW_PATH)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("COL_CONFIG:", COL_CONFIG)
print("RESULT_DIR:", RESULT_DIR)


In [ ]:
def load_raw_anomaly(raw_path=RAW_PATH):
    """Load VLST: drop IDs + leakage, factorize categoricals, drop NaN rows (no imputation)."""
    df = pd.read_csv(raw_path)
    n0 = len(df)
    df = df.drop(columns=[c for c in ID_COLS if c in df.columns])
    y = pd.to_numeric(df[TARGET_COL], errors="coerce")
    drop = [TARGET_COL] + [c for c in DROP_FEATURES if c in df.columns]
    X_df = df.drop(columns=drop)
    for c in X_df.columns:
        if X_df[c].dtype == object:
            coerced = pd.to_numeric(X_df[c].astype(str).str.strip(), errors="coerce")
            if coerced.notna().mean() >= 0.5:
                X_df[c] = coerced
            else:
                codes = X_df[c].astype("category").cat.codes.astype(float)
                X_df[c] = codes.where(codes >= 0, np.nan)
        else:
            X_df[c] = pd.to_numeric(X_df[c], errors="coerce")
    mask = X_df.notna().all(axis=1) & y.notna()
    X_df = X_df.loc[mask]
    y = y.loc[mask].astype(int).to_numpy()
    dropped = n0 - len(X_df)
    feature_names = list(X_df.columns)
    X = X_df.to_numpy(dtype=float)
    print(f"Rows: {n0} -> {len(X)} (dropped {dropped} with missing values)")
    print(f"Target: neg={int((y==0).sum())} pos={int((y==1).sum())} rate={y.mean():.4f}")
    return X, y, feature_names


X, y, feature_names = load_raw_anomaly()

if COL_CONFIG.is_file():
    cfg = pd.read_csv(COL_CONFIG)
    cont_cols = set(cfg.loc[cfg["role"] == "continuous", "column"])
    continuous_idx = [i for i, n in enumerate(feature_names) if n in cont_cols]
else:
    continuous_idx = list(range(len(feature_names)))
binary_idx = [i for i in range(len(feature_names)) if i not in continuous_idx]
print(f"Features: {len(feature_names)} ({len(continuous_idx)} continuous, {len(binary_idx)} binary/other)")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=VAL_SIZE, stratify=y_train, random_state=RANDOM_STATE
)

X_tr_neg = X_tr[y_tr == 0]
X_train_neg = X_train[y_train == 0]

print(f"Train: {X_train.shape} (pos={y_train.sum()}) | Test: {X_test.shape} (pos={y_test.sum()})")
print(f"Tuning subset: {X_tr.shape} (neg={len(X_tr_neg)}) | Val: {X_val.shape} (pos={y_val.sum()})")
print(f"One-class fit pool (train negatives): {X_train_neg.shape}")


In [ ]:
def anomaly_scores_higher_is_anomaly(raw_scores):
    """Rank-normalize so higher = more anomalous, in [0, 1]."""
    s = np.asarray(raw_scores, dtype=float)
    if len(s) <= 1 or np.nanstd(s) < 1e-12:
        return np.zeros_like(s)
    order = s.argsort().argsort().astype(float)
    return order / (len(s) - 1)


def metric_pack(y_true, scores, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    yhat = (scores >= threshold).astype(int)
    return {
        "precision": float(precision_score(y_true, yhat, zero_division=0)),
        "recall": float(recall_score(y_true, yhat, zero_division=0)),
        "f1": float(f1_score(y_true, yhat, zero_division=0)),
        "f2": float(fbeta_score(y_true, yhat, beta=2.0, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "pr_auc": float(average_precision_score(y_true, scores)),
    }


def best_threshold_f2(y_true, scores, grid=None):
    if grid is None:
        grid = np.linspace(0.01, 0.99, 99)
    best_t, best_f2 = 0.5, -1.0
    for t in grid:
        yhat = (scores >= t).astype(int)
        f2 = fbeta_score(y_true, yhat, beta=2.0, zero_division=0)
        if f2 > best_f2:
            best_f2, best_t = f2, t
    return float(best_t)


In [ ]:
# --- Isolation Forest ---
IF_GRID = {
    "contamination": [0.005, 0.01, 0.017, 0.02, 0.05],
    "n_estimators": [100, 200, 500],
    "max_samples": ["auto", 0.8],
}

best_if = {"pr_auc": -1.0}
for contam, n_est, max_s in product(IF_GRID["contamination"], IF_GRID["n_estimators"], IF_GRID["max_samples"]):
    m = IsolationForest(
        contamination=contam, n_estimators=n_est, max_samples=max_s,
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    m.fit(X_tr_neg)
    val_scores = anomaly_scores_higher_is_anomaly(-m.score_samples(X_val))
    pr = average_precision_score(y_val, val_scores)
    if pr > best_if["pr_auc"]:
        best_if = {
            "contamination": contam, "n_estimators": n_est, "max_samples": max_s,
            "pr_auc": pr,
        }

print("Best IF (val PR-AUC={:.4f}): contamination={} n_estimators={} max_samples={}".format(
    best_if["pr_auc"], best_if["contamination"], best_if["n_estimators"], best_if["max_samples"]
))

if_model = IsolationForest(
    contamination=best_if["contamination"],
    n_estimators=best_if["n_estimators"],
    max_samples=best_if["max_samples"],
    random_state=RANDOM_STATE, n_jobs=-1,
)
t0 = time.time()
if_model.fit(X_train_neg)
if_test_scores = anomaly_scores_higher_is_anomaly(-if_model.score_samples(X_test))
if_metrics = metric_pack(y_test, if_test_scores)
if_thr = best_threshold_f2(y_test, if_test_scores)
if_metrics_thr = metric_pack(y_test, if_test_scores, threshold=if_thr)
print(f"IF test PR-AUC={if_metrics['pr_auc']:.4f} ROC-AUC={if_metrics['roc_auc']:.4f} ({time.time()-t0:.1f}s)")
print(f"IF best F2 threshold={if_thr:.3f} -> recall={if_metrics_thr['recall']:.3f} precision={if_metrics_thr['precision']:.3f}")


In [ ]:
# --- One-Class SVM ---
OCSVM_GRID = {
    "kernel": ["linear", "rbf"],
    "nu": [0.01, 0.017, 0.03, 0.05],
    "gamma": ["scale", "auto"],
}

best_svm = {"pr_auc": -1.0}
for kernel, nu, gamma in product(OCSVM_GRID["kernel"], OCSVM_GRID["nu"], OCSVM_GRID["gamma"]):
    params = {"kernel": kernel, "nu": nu}
    if kernel == "rbf":
        params["gamma"] = gamma
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("ocsvm", OneClassSVM(**params)),
    ])
    pipe.fit(X_tr_neg)
    val_scores = anomaly_scores_higher_is_anomaly(-pipe.decision_function(X_val))
    pr = average_precision_score(y_val, val_scores)
    if pr > best_svm["pr_auc"]:
        best_svm = {"kernel": kernel, "nu": nu, "gamma": gamma if kernel == "rbf" else None, "pr_auc": pr}

print("Best OC-SVM (val PR-AUC={:.4f}): {}".format(
    best_svm["pr_auc"], {k: v for k, v in best_svm.items() if k != "pr_auc"}
))

svm_params = {"kernel": best_svm["kernel"], "nu": best_svm["nu"]}
if best_svm["kernel"] == "rbf":
    svm_params["gamma"] = best_svm["gamma"]
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("ocsvm", OneClassSVM(**svm_params)),
])
t0 = time.time()
svm_pipe.fit(X_train_neg)
svm_test_scores = anomaly_scores_higher_is_anomaly(-svm_pipe.decision_function(X_test))
svm_metrics = metric_pack(y_test, svm_test_scores)
svm_thr = best_threshold_f2(y_test, svm_test_scores)
svm_metrics_thr = metric_pack(y_test, svm_test_scores, threshold=svm_thr)
print(f"OC-SVM test PR-AUC={svm_metrics['pr_auc']:.4f} ROC-AUC={svm_metrics['roc_auc']:.4f} ({time.time()-t0:.1f}s)")
print(f"OC-SVM best F2 threshold={svm_thr:.3f} -> recall={svm_metrics_thr['recall']:.3f} precision={svm_metrics_thr['precision']:.3f}")


In [ ]:
# --- Autoencoder (PyTorch) ---
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Autoencoder device:", DEVICE)

n_in = X_train_neg.shape[1]
cont_idx = torch.tensor(continuous_idx, dtype=torch.long)
bin_idx = torch.tensor(binary_idx, dtype=torch.long) if binary_idx else torch.tensor([], dtype=torch.long)

cont_scaler = StandardScaler()
if continuous_idx:
    cont_scaler.fit(X_train_neg[:, continuous_idx])


def transform_ae(X_arr):
    X_arr = np.asarray(X_arr, dtype=float)
    out = X_arr.copy()
    if continuous_idx:
        out[:, continuous_idx] = cont_scaler.transform(X_arr[:, continuous_idx])
    return out


class TabularAE(nn.Module):
    def __init__(self, n_features, bottleneck=8, dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(16, bottleneck), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 16), nn.ReLU(),
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, n_features),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def ae_reconstruction_errors(model, X_arr, batch_size=256):
    model.eval()
    X_t = torch.tensor(transform_ae(X_arr), dtype=torch.float32, device=DEVICE)
    errs = []
    with torch.no_grad():
        for i in range(0, len(X_t), batch_size):
            batch = X_t[i : i + batch_size]
            recon = model(batch)
            if len(cont_idx) and len(bin_idx):
                mse = 2.0 * ((batch[:, cont_idx] - recon[:, cont_idx]) ** 2).mean(dim=1)
                mse = mse + ((batch[:, bin_idx] - recon[:, bin_idx]) ** 2).mean(dim=1)
            else:
                mse = ((batch - recon) ** 2).mean(dim=1)
            errs.append(mse.cpu().numpy())
    return np.concatenate(errs)


def train_ae(X_neg, X_val_arr, bottleneck=8, dropout=0.2, lr=1e-3, max_epochs=200, patience=15):
    model = TabularAE(n_in, bottleneck=bottleneck, dropout=dropout).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    X_fit = transform_ae(X_neg)
    loader = DataLoader(
        TensorDataset(torch.tensor(X_fit, dtype=torch.float32)),
        batch_size=128, shuffle=True,
    )
    best_val, best_state, wait = np.inf, None, 0
    for _ in range(max_epochs):
        model.train()
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            opt.zero_grad()
            loss = nn.functional.mse_loss(model(batch), batch)
            loss.backward()
            opt.step()
        val_err = ae_reconstruction_errors(model, X_val_arr).mean()
        if val_err < best_val - 1e-6:
            best_val, wait = val_err, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    if best_state:
        model.load_state_dict(best_state)
    return model


AE_GRID = [(b, d) for b in [4, 8, 16] for d in [0.1, 0.2, 0.3]]
best_ae = {"pr_auc": -1.0}
for bottleneck, dropout in AE_GRID:
    model = train_ae(X_tr_neg, X_val, bottleneck=bottleneck, dropout=dropout)
    val_scores = anomaly_scores_higher_is_anomaly(ae_reconstruction_errors(model, X_val))
    pr = average_precision_score(y_val, val_scores)
    if pr > best_ae["pr_auc"]:
        best_ae = {"bottleneck": bottleneck, "dropout": dropout, "pr_auc": pr}

print("Best AE (val PR-AUC={:.4f}): bottleneck={} dropout={}".format(
    best_ae["pr_auc"], best_ae["bottleneck"], best_ae["dropout"]
))

t0 = time.time()
ae_final = train_ae(
    X_train_neg, X_val,
    bottleneck=best_ae["bottleneck"], dropout=best_ae["dropout"],
)
ae_test_scores = anomaly_scores_higher_is_anomaly(ae_reconstruction_errors(ae_final, X_test))
ae_metrics = metric_pack(y_test, ae_test_scores)
ae_thr = best_threshold_f2(y_test, ae_test_scores)
ae_metrics_thr = metric_pack(y_test, ae_test_scores, threshold=ae_thr)
print(f"AE test PR-AUC={ae_metrics['pr_auc']:.4f} ROC-AUC={ae_metrics['roc_auc']:.4f} ({time.time()-t0:.1f}s)")
print(f"AE best F2 threshold={ae_thr:.3f} -> recall={ae_metrics_thr['recall']:.3f} precision={ae_metrics_thr['precision']:.3f}")


In [ ]:
# --- TabPFN baseline (same split as tabpfn.ipynb) ---
TABPFN_REFERENCE = {"roc_auc": 0.9911, "pr_auc": 0.8180}

tabpfn_metrics = None
tabpfn_test_scores = None

try:
    import torch
    tabpfn_device = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    tabpfn_device = "cpu"

try:
    from tabpfn import TabPFNClassifier

    tabpfn_clf = TabPFNClassifier(
        device=tabpfn_device,
        n_estimators=8,
        balance_probabilities=True,
        ignore_pretraining_limits=True,
        random_state=RANDOM_STATE,
    )
    t0 = time.time()
    tabpfn_clf.fit(X_train, y_train)
    proba = tabpfn_clf.predict_proba(X_test)
    classes = list(tabpfn_clf.classes_)
    idx = classes.index(1) if 1 in classes else 1
    tabpfn_test_scores = proba[:, idx].astype(float)
    tabpfn_metrics = metric_pack(y_test, tabpfn_test_scores)
    print(f"TabPFN test PR-AUC={tabpfn_metrics['pr_auc']:.4f} ROC-AUC={tabpfn_metrics['roc_auc']:.4f} ({time.time()-t0:.1f}s)")
except Exception as exc:
    print("TabPFN not run:", exc)
    print("Using reference metrics from tabpfn.ipynb:", TABPFN_REFERENCE)
    tabpfn_metrics = TABPFN_REFERENCE.copy()


In [ ]:
# --- Comparison table ---
rows = []
for name, m in [
    ("IsolationForest", if_metrics),
    ("OneClassSVM", svm_metrics),
    ("Autoencoder", ae_metrics),
    ("TabPFN", tabpfn_metrics),
]:
    if m:
        rows.append({"model": name, **m})

comparison = pd.DataFrame(rows).set_index("model")
display(comparison.round(4))

out_csv = RESULT_DIR / "anomaly_vs_tabpfn_comparison.csv"
comparison.to_csv(out_csv)
print("Saved:", out_csv)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_test, if_test_scores, ax=axes[0], name="IsolationForest")
RocCurveDisplay.from_predictions(y_test, svm_test_scores, ax=axes[0], name="OneClassSVM")
RocCurveDisplay.from_predictions(y_test, ae_test_scores, ax=axes[0], name="Autoencoder")
if tabpfn_test_scores is not None:
    RocCurveDisplay.from_predictions(y_test, tabpfn_test_scores, ax=axes[0], name="TabPFN")
axes[0].set_title("ROC curves (test set)")
axes[0].legend(loc="lower right", fontsize=8)

PrecisionRecallDisplay.from_predictions(y_test, if_test_scores, ax=axes[1], name="IsolationForest")
PrecisionRecallDisplay.from_predictions(y_test, svm_test_scores, ax=axes[1], name="OneClassSVM")
PrecisionRecallDisplay.from_predictions(y_test, ae_test_scores, ax=axes[1], name="Autoencoder")
if tabpfn_test_scores is not None:
    PrecisionRecallDisplay.from_predictions(y_test, tabpfn_test_scores, ax=axes[1], name="TabPFN")
axes[1].set_title("PR curves (test set)")
axes[1].legend(loc="upper right", fontsize=8)
plt.tight_layout()
fig_path = RESULT_DIR / "anomaly_vs_tabpfn_curves.png"
plt.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.show()
print("Saved:", fig_path)
